# Workshop: sparse autoencoders

Last week we looked at an autoencoder on MNIST with a two-dimensional latent layer, and saw that the ten digits end up in different parts of the latent plane. Nobody told the network about digits: it only tried to reconstruct images. The digits are *in there somewhere*, but written in the network's own code, as two numbers $z_1$ and $z_2$ that mean nothing to us one at a time.

This is the central problem of **interpretability**. A trained network holds information in its hidden layers, and we would like to read it. A **sparse autoencoder (SAE)** is a tool for doing that. It is a second, small autoencoder that we train on the *activations* of a layer of the first network. It rewrites each activation vector as a combination of a larger number of **features**, with the rule that only a few features may be switched on at a time. The hope is that each feature then stands for one recognisable thing. It is sort of the backwards of a normal autoencoder, usually the latent layer is smaller than the input forcing it to extract features, in a sparse autoencoder the latent layer is larger but with some extra rule that says only a few of the neurons should be on at any one time so if there are features, they correspond to a single node.

In this workshop you will build an SAE, train it on the latent layer of the MNIST autoencoder, and see how far it gets towards finding the ten digits.

### An honest warning about scale

SAEs were invented for very large layers: the hidden layers of language models, with thousands of dimensions, on which people train SAEs with tens of thousands or millions of features. Our layer has two dimensions and our SAE will have ten features. That makes this a toy, and a slightly unfair one, for a reason that Section 6 explains: a high-dimensional space has room for a huge number of directions that are *nearly* at right angles to one another, and a plane does not. The advantage of the toy is that we can draw everything. Keep in mind throughout that the real thing happens in spaces we cannot draw.

### What you will practise

- pulling the activations of one layer out of a trained network;
- writing a sparse autoencoder: encoder, feature activations, decoder;
- two ways of making the features sparse, an **L1 penalty** and a **TopK** rule, and why each one works;
- measuring an SAE: reconstruction error, number of active features, dead features;
- interpreting features, and recognising what an SAE can and cannot find.

| Section | Topic | Rough time |
|---|---|---|
| 0 | Why sparse autoencoders? | 10 min (reading) |
| 1 | The MNIST autoencoder, and extracting its middle layer | 15 min |
| 2 | Anatomy of a sparse autoencoder | 20 min |
| 3 | Sparsity from an L1 penalty | 25 min |
| 4 | Sparsity from TopK | 15 min |
| 5 | What did the features learn? | 25 min |
| 6 | Why the real thing is different: high dimensions | 10 min |

Cells marked **Exercise** ask you to complete some code. Cells marked **Question** ask for a sentence or two; write your answer in the cell underneath.

## 0. Why sparse autoencoders?

### Neurons are hard to read

The obvious way to understand a hidden layer is to look at its units one at a time: find the inputs that make unit 37 large, and guess what unit 37 "means". This usually fails. A single unit typically responds to several unrelated things, and a single recognisable thing is typically spread across many units. Our own autoencoder shows this already: neither $z_1$ nor $z_2$ means "this is a 7". The digit is encoded by *where the point is in the plane*, which involves both coordinates together. In the jargon it is a dense representation.

### Features as directions

The working assumption behind SAEs is that a layer represents many separate **features** of its input (for a language model: "this is about cooking", "this word is a plural", "we are inside a quotation"), and that each feature corresponds to a **direction** in the layer's activation space. If a feature is present, the activation vector moves some distance along that feature's direction. The whole activation vector is then a sum
$$x \;\approx\; b + f_1 d_1 + f_2 d_2 + \dots + f_m d_m ,$$
where $d_j$ is the direction of feature $j$, the number $f_j \ge 0$ says how strongly feature $j$ is present, and $b$ is a fixed offset. The units of the layer are just the coordinate axes we happen to be using, and there is no reason for the feature directions to line up with them.

### More features than dimensions

Here is the important twist. There can be many more features than the layer has dimensions: $m$ can be much larger than $d$. This sounds impossible, since $d$ numbers cannot hold $m$ independent quantities. It becomes possible if the features are **sparse**, meaning that for any one input almost all of the $f_j$ are zero. A sentence is about cooking or about astronomy, rarely both. If only a handful of features are active at once, the network can get away with packing many feature directions into a few dimensions. This packing is called **superposition**.

### What the SAE does

A sparse autoencoder tries to undo the packing. We give it activation vectors $x$ and ask it to find directions $d_j$ and activations $f_j$ so that the sum above reconstructs $x$, using as few non-zero $f_j$ as it can. It has two parts.

- An **encoder** that reads $x$ and decides how active each feature is.
- A **decoder** that rebuilds $x$ from the feature activations. The decoder is nothing more than the sum above: its weights are the feature directions.

It is an autoencoder because it is trained to reproduce its own input. But compare it with the MNIST autoencoder:

| | MNIST autoencoder | sparse autoencoder |
|---|---|---|
| input | an image (784 numbers) | the activations of a layer (here 2 numbers) |
| middle layer | **narrower** than the input (2) | **wider** than the input (here 10) |
| what forces it to learn something | the bottleneck | the sparsity |
| what we want from it | a compact code | a readable code |

The MNIST autoencoder squeezes; the sparse autoencoder spreads out. A wide middle layer with no constraint would be pointless, since the network could simply copy its input across. The sparsity constraint is what makes the wide layer meaningful, and most of this workshop is about how to impose it.

## 1. The MNIST autoencoder, and extracting its middle layer

The next cells are the autoencoder from the lecture: the same 784-512-256-2-256-512-784 network, trained in the same way. If the file `ae_mnist.pt` is in the folder (it is supplied with this notebook) the trained weights are loaded from it, which takes no time. If you delete that file the network is trained again from scratch, which takes a few minutes; your latent space will then differ from the one described in the answers, because the layout of the latent plane depends on the details of training.

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ---------------------------------------------------------------- settings
LATENT = 2
EPOCHS = 30
BATCH = 256
LR = 1e-3
SEED = 0
MODEL_FILE = "ae_mnist.pt"   # delete this file to force retraining

torch.manual_seed(SEED)
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")

# ---------------------------------------------------------------- data
tf = transforms.ToTensor()  # pixels in [0, 1]
train = datasets.MNIST("data", train=True, download=True, transform=tf)
test = datasets.MNIST("data", train=False, download=True, transform=tf)
train_loader = DataLoader(train, batch_size=BATCH, shuffle=True)

In [ ]:
# ---------------------------------------------------------------- model
class AutoEncoder(nn.Module):
    def __init__(self, latent=LATENT):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, latent),            # linear bottleneck
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 784),               # logits; sigmoid gives pixels
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    @torch.no_grad()
    def decode_images(self, z):
        return torch.sigmoid(self.decoder(z)).view(-1, 28, 28)


model = AutoEncoder().to(device)

# ---------------------------------------------------------------- train or load
if os.path.exists(MODEL_FILE):
    model.load_state_dict(torch.load(MODEL_FILE, map_location=device))
    print(f"loaded {MODEL_FILE}")
else:
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss(reduction="sum")  # sharper digits than MSE
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            logits, _ = model(x)
            loss = loss_fn(logits, x.view(x.size(0), -1)) / x.size(0)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * x.size(0)
        print(f"epoch {epoch:3d}   loss per image {total / len(train):.2f}")
    torch.save(model.state_dict(), MODEL_FILE)
    print(f"saved {MODEL_FILE}")

model.eval();

### Extracting the middle layer

From now on the autoencoder is **frozen**. We will never change its weights again. We only want to record what its middle layer does when it is shown images.

To train an SAE on a layer we need a data set of that layer's activations: run many inputs through the network, and for each one write down the activation vector of the layer we care about. For us that layer is the two-dimensional latent layer, the output of `model.encoder`. So the recipe is short:

1. take the images, in a fixed order and without shuffling;
2. pass them through `model.encoder`, inside `torch.no_grad()` because we are not training;
3. collect the results into one array with a row for each image.

**Exercise 1.1.** Complete the function below. It should return a tensor of shape `(number of images, 2)`.

In [ ]:
# Exercise 1.1
@torch.no_grad()
def extract_latents(dataset):
    # images as one big tensor of shape (N, 1, 28, 28), values in [0, 1]
    images = dataset.data.float().div(255).unsqueeze(1)
    out = []
    for i in range(0, len(images), 1000):          # 1000 images at a time
        x = images[i:i + 1000].to(device)
        z = ...                                     # the latent activations for this batch
        out.append(z.cpu())
    return torch.cat(out)


Z_train = extract_latents(train)
Z_test = extract_latents(test)
Y_train = train.targets.numpy()
Y_test = test.targets.numpy()
print(Z_train.shape, Z_test.shape)

In our network the layer we want happens to be the output of a named part of the model, so we could just call that part. In a large model the layer of interest is usually buried in the middle, and the standard tool is a **forward hook**: a function that PyTorch calls every time a chosen layer produces its output, which we use to save a copy. The next cell does the same extraction with a hook, to show the pattern. This is how activations are collected from real models.

In [ ]:
saved = []

def save_output(module, inputs, output):
    saved.append(output.detach().cpu())

layer = model.encoder[-1]                        # the last layer of the encoder: Linear(256, 2)
handle = layer.register_forward_hook(save_output)
with torch.no_grad():
    model(test.data[:1000].float().div(255).unsqueeze(1).to(device))   # run the whole model; ignore its output
handle.remove()                                  # always remove a hook when you have finished

print("hook gives the same activations:", torch.allclose(saved[0], Z_test[:1000], atol=1e-5))

Here is the latent space, as you saw it in the lecture. Each point is one test image, coloured by its digit. The labels are used only for colouring; neither the autoencoder nor the SAE ever sees them.

In [ ]:
def latent_scatter(ax, Z, Y, lim=None, s=2, labels=True):
    Z = np.asarray(Z)
    ax.scatter(Z[:, 0], Z[:, 1], c=Y, cmap="tab10", s=s, alpha=0.6, vmin=-0.5, vmax=9.5, rasterized=True)
    if labels:
        for d in range(10):
            c = np.median(Z[Y == d], axis=0)
            ax.text(c[0], c[1], str(d), fontsize=12, weight="bold", ha="center", va="center",
                    bbox=dict(boxstyle="circle,pad=0.15", fc="white", alpha=0.85))
    if lim is not None:
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
latent_scatter(axes[0], Z_test, Y_test, lim=32)
axes[0].set_title("Latent space (test set)")
latent_scatter(axes[1], Z_test, Y_test, lim=7, s=5)
axes[1].set_title("The middle, enlarged")
plt.tight_layout()

**Question 1.1.** Look carefully at both panels. (a) Which digits stretch out from the middle in a direction of their own? (b) Find two digits that lie in roughly the *same direction* from the middle, one close in and one further out. (c) Which digits are crowded together near the middle? Your answers will matter in Section 5.

*Your answer:*

### Rescaling

The size of the latent coordinates is arbitrary: the autoencoder could have used numbers ten times larger or smaller and worked just as well. Before training an SAE it is standard to shift and rescale the activations so that they are centred and have a typical length of one. Then a reconstruction error of, say, 0.05 means the same thing whatever layer the activations came from, and the strength of the sparsity penalty has a consistent meaning.

We subtract the mean of the training activations and divide by a single number, chosen so that the average squared length of a centred training vector is one. We divide both coordinates by the *same* number, so the shape of the latent space is unchanged.

In [ ]:
z_mean = Z_train.mean(dim=0)
z_scale = (Z_train - z_mean).pow(2).sum(dim=1).mean().sqrt()

X_train = (Z_train - z_mean) / z_scale        # what the SAE is trained on
X_test = (Z_test - z_mean) / z_scale

def to_latent(x):
    # undo the rescaling: from SAE units back to the autoencoder's latent units
    return x * z_scale + z_mean

print("mean:", z_mean.numpy().round(2), "  scale:", round(z_scale.item(), 2))
print("average squared length after rescaling:", X_train.pow(2).sum(dim=1).mean().item())

## 2. Anatomy of a sparse autoencoder

Write $x$ for one rescaled activation vector, with $d$ entries. For us $d = 2$. The SAE has $m$ features; we take $m = 10$, hoping for one feature for each digit. Its parameters are

- an encoder matrix $W_{\text{enc}}$ of shape $m \times d$ and an encoder bias $b_{\text{enc}}$ with $m$ entries;
- a decoder matrix $W_{\text{dec}}$ of shape $d \times m$ and a decoder bias $b_{\text{dec}}$ with $d$ entries.

The **encoder** computes the feature activations:
$$f = \mathrm{ReLU}\big(W_{\text{enc}}\,(x - b_{\text{dec}}) + b_{\text{enc}}\big).$$
Each feature takes a weighted sum of the input and passes it through a ReLU, so every $f_j$ is zero or positive. A feature whose activation is zero is **off**; a feature whose activation is positive is **active**.

The **decoder** rebuilds the input:
$$\hat x = W_{\text{dec}}\, f + b_{\text{dec}} \;=\; b_{\text{dec}} + \sum_{j=1}^{m} f_j\, d_j ,$$
where $d_j$ is column $j$ of $W_{\text{dec}}$. This is exactly the picture from Section 0. The reconstruction starts at the point $b_{\text{dec}}$ and, for each active feature, moves a distance $f_j$ along that feature's direction $d_j$. So:

- $b_{\text{dec}}$ is the **origin**, the point that is reconstructed when no feature is active. The encoder subtracts it first, so that features respond to the position of $x$ *relative to the origin*.
- the columns of $W_{\text{dec}}$ are the **feature directions**. We will keep each of them at length 1, for a reason that comes up in Section 3.

There is no hidden layer and no further non-linearity: one linear map and a ReLU going in, one linear map coming out. The simplicity is deliberate. Because the decoder is linear, each feature has one fixed direction, and "what feature $j$ contributes" has a clear meaning.

**Exercise 2.1.** Complete `encode` and `decode` in the class below, following the two formulas. Remember that in PyTorch a batch of inputs is a matrix with one *row* for each input, so for a batch `x` of shape `(N, d)` the product $W_{\text{enc}}(x - b_{\text{dec}})$ is written `(x - self.b_dec) @ self.W_enc.T`.

In [ ]:
# Exercise 2.1
class SparseAutoEncoder(nn.Module):
    def __init__(self, d_in=2, n_features=10):
        super().__init__()
        W = F.normalize(torch.randn(d_in, n_features), dim=0)   # random directions of length 1
        self.W_dec = nn.Parameter(W)                    # (d_in, n_features): columns are feature directions
        self.W_enc = nn.Parameter(W.T.clone())          # (n_features, d_in): start as the transpose
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        self.b_dec = nn.Parameter(torch.zeros(d_in))

    def pre_activations(self, x):
        # the weighted sums, before the ReLU: shape (N, n_features)
        return ...

    def encode(self, x):
        # feature activations f: shape (N, n_features)
        return ...

    def decode(self, f):
        # reconstruction x_hat: shape (N, d_in)
        return ...

    def forward(self, x):
        f = self.encode(x)
        return self.decode(f), f

This cell checks the shapes. It should print `torch.Size([5, 2]) torch.Size([5, 10])` and `True`.

In [ ]:
torch.manual_seed(0)
sae_check = SparseAutoEncoder()
x_hat, f = sae_check(X_train[:5])
print(x_hat.shape, f.shape)
print(bool((f >= 0).all()))

### Training, and three numbers to watch

The training loop is given below. It is an ordinary loop, with two details that are particular to SAEs.

- The loss is computed by a function `loss_fn(x, x_hat, f)` that you will write. It receives the input, the reconstruction and the feature activations.
- After every step, each column of `W_dec` is rescaled to length 1.

To judge an SAE we will use three numbers, computed on the test set.

- **Reconstruction error.** The mean squared distance between $x$ and $\hat x$, divided by the mean squared distance of $x$ from the average. This is the *fraction of variance unexplained*: 0 means perfect reconstruction, and 1 means the SAE does no better than always answering with the average.
- **L0.** The average number of features that are active for an input. ("L0" is the name for the count of non-zero entries in a vector.) This is what we mean by sparsity: the smaller, the sparser.
- **Dead features.** The number of features that are never active on any input. A dead feature is wasted.

**Exercise 2.2.** Complete the three lines in `evaluate`.

In [ ]:
def train_sae(sae, loss_fn, X=X_train, epochs=30, batch=512, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(sae.parameters(), lr=lr)
    for epoch in range(epochs):
        order = torch.randperm(len(X))
        for i in range(0, len(X), batch):
            x = X[order[i:i + batch]]
            x_hat, f = sae(x)
            loss = loss_fn(x, x_hat, f)
            opt.zero_grad()
            loss.backward()
            opt.step()
            with torch.no_grad():                       # keep every feature direction at length 1
                sae.W_dec.data = F.normalize(sae.W_dec.data, dim=0)
    return sae

In [ ]:
# Exercise 2.2
@torch.no_grad()
def evaluate(sae, X=X_test):
    x_hat, f = sae(X)
    error = ...        # mean squared distance between X and x_hat, divided by
                       # the mean squared distance between X and its mean
                       # hint: (X - x_hat).pow(2).sum(dim=1).mean()
    l0 = ...           # average number of active features per input
                       # hint: (f > 0) is a table of True/False; .float() turns it into 1/0
    dead = ...         # number of features that are not active for any input
    return {"error": float(error), "L0": float(l0), "dead": int(dead)}

### An SAE with no sparsity

First train the SAE with reconstruction as the only goal, so that we can see what goes wrong.

In [ ]:
def reconstruction_loss(x, x_hat, f):
    return (x - x_hat).pow(2).sum(dim=1).mean()

torch.manual_seed(0)
sae_plain = train_sae(SparseAutoEncoder(), reconstruction_loss)
print(evaluate(sae_plain))

**Question 2.1.** (a) Is the reconstruction good? (b) How many of the ten features are active for a typical input? (c) Explain why this SAE, though it reconstructs well, is useless for interpretation. *Hint: the input has two dimensions. How many features would you need to reconstruct it exactly, and is there any reason for the SAE to prefer one way of doing it over another?*

*Your answer:*

## 3. Sparsity from an L1 penalty

We want few features to be active at once: the count of active nodes is called L0 and so we want L0 to be small. The direct approach would be to add L0 to the loss. That does not work, precisely because L0 is a count. It does not change at all when an activation gets a little smaller, and then jumps when the activation reaches zero. Its gradient is zero almost everywhere, so gradient descent gets no signal from it.

The classic substitute is the **L1 penalty**: add the *sum of the activations* to the loss,
$$\mathcal L \;=\; \underbrace{\frac1N \sum_{n} \lVert x_n - \hat x_n\rVert^2}_{\text{reconstruct well}} \;+\; \lambda\, \underbrace{\frac1N \sum_n \sum_{j} f_{nj}}_{\text{use little activation}} .$$
The L1 norm of a vector is the sum of the absolute values of its entries. Our activations are never negative, so we can leave the absolute value out. The number $\lambda \ge 0$ sets how much we care about sparsity compared with reconstruction.

### Why should penalising the *sum* produce *zeros*?

This is the key idea of the section, and it is not obvious. A penalty on the size of the activations might be expected to make them all a bit smaller, not to switch most of them off. To see what really happens, strip the problem down to a single activation.

Suppose that, to reconstruct perfectly, a feature ought to have activation $a$. We are free to choose a different activation $f \ge 0$, and we pay $(a - f)^2$ for the reconstruction error this causes, plus a penalty. Compare two penalties:

- **L1:** minimise $(a - f)^2 + \lambda f$;
- **L2:** minimise $(a - f)^2 + \lambda f^2$.

**Exercise 3.1.** For each penalty, find the best $f$ for every value of $a$ on a grid, by brute force: evaluate the cost for a fine grid of candidate values of $f$ and take the smallest. Plot the best $f$ against $a$ for both penalties, with $\lambda = 1$.

In [ ]:
# Exercise 3.1
lam = 1.0
a_values = np.linspace(0, 2, 201)             # the activation that would reconstruct perfectly
f_grid = np.linspace(0, 2, 2001)              # candidate activations to try

best_l1, best_l2 = [], []
for a in a_values:
    cost_l1 = ...                             # cost of every candidate f, with the L1 penalty
    cost_l2 = ...                             # the same with the L2 penalty
    best_l1.append(f_grid[np.argmin(cost_l1)])
    best_l2.append(f_grid[np.argmin(cost_l2)])

plt.figure(figsize=(5.5, 4))
plt.plot(a_values, a_values, "k:", label="no penalty")
plt.plot(a_values, best_l1, label="L1 penalty")
plt.plot(a_values, best_l2, label="L2 penalty")
plt.xlabel("activation needed for perfect reconstruction, $a$")
plt.ylabel("best activation, $f$")
plt.legend();

**Question 3.1.** (a) Describe the two curves. For which values of $a$ does each penalty give exactly $f = 0$? (b) Confirm your picture with calculus: set the derivative of each cost to zero and solve for $f$, remembering that $f$ cannot be negative. (c) Explain in words why L1 produces zeros and L2 does not. *Hint: compare how hard each penalty pushes on an activation that is already small.* (d) The L1 curve also shows a cost of the method. What happens to activations that are *not* set to zero?

*Your answer:*

The function $f = \max(a - \lambda/2, 0)$ is called **soft thresholding**. It is the whole mechanism of L1 sparsity in one line: activations that would be weak are switched off entirely, and the rest are kept a little reduced. Inside the SAE the same thing happens, through the ReLU and the encoder bias: training drives $b_{\text{enc}}$ negative, so that a feature's weighted sum has to exceed a threshold before the feature switches on.

### Why the feature directions must have length 1

There is a loophole in the L1 penalty. The reconstruction is a sum of terms $f_j d_j$. If we halve every activation $f_j$ and double the length of every direction $d_j$, each product $f_j d_j$ is unchanged, so the reconstruction is exactly as good, but the penalty $\lambda \sum f_j$ has been halved. Repeating the trick would make the penalty as small as we like without making anything sparser. Fixing the length of every $d_j$ at 1 closes the loophole: an activation of $f_j$ then always means "move a distance $f_j$", and the penalty measures something real. That is what the last line of `train_sae` is for.

### Training with the L1 penalty

**Exercise 3.2.** Write a function `make_l1_loss(lam)` that returns a loss function for a given $\lambda$. Then train an SAE for each value of $\lambda$ in the list, evaluate each one, and collect the results in the dictionary `l1_results`.

In [ ]:
# Exercise 3.2
def make_l1_loss(lam):
    def loss_fn(x, x_hat, f):
        reconstruction = ...            # as in reconstruction_loss above
        sparsity = ...                  # sum of the activations for each input, averaged over the batch
        return reconstruction + lam * sparsity
    return loss_fn


lambdas = [0.0, 0.03, 0.1, 0.2, 0.3, 0.45, 0.6, 0.8, 1.0]
l1_saes, l1_results = {}, {}
for lam in lambdas:
    torch.manual_seed(0)
    sae = ...                           # a new SparseAutoEncoder, trained with make_l1_loss(lam)
    l1_saes[lam] = sae
    l1_results[lam] = evaluate(sae)
    print(f"lambda = {lam:<5}", l1_results[lam])

The usual way to display these results is to plot reconstruction error against L0, with one point for each SAE.

In [ ]:
def tradeoff_plot(results, label, marker="o"):
    l0 = [r["L0"] for r in results.values()]
    err = [r["error"] for r in results.values()]
    plt.plot(l0, err, marker=marker, label=label)
    for key, r in results.items():
        plt.annotate(str(key), (r["L0"], r["error"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
    plt.xlabel("L0 (average number of active features)")
    plt.ylabel("reconstruction error")

plt.figure(figsize=(6, 4))
tradeoff_plot(l1_results, "L1 penalty (labels are $\\lambda$)")
plt.legend();

**Question 3.2.** (a) Describe what happens to L0 and to the reconstruction error as $\lambda$ increases. (b) Why can't we have both a small L0 and a small error? (c) For the largest values of $\lambda$, L0 is below 1. What does that mean for some of the inputs, and what are they reconstructed as?

*Your answer:*

## 4. Sparsity from TopK

The L1 penalty gets sparsity indirectly. We choose $\lambda$, train, and find out afterwards how many features are active. It also shrinks the activations it keeps. **TopK** is a more direct idea.

> Compute the weighted sums for all the features as before. Keep the $k$ largest, and set all the others to zero.

That is the whole method. Exactly $k$ features are active for every input, because we said so. There is no penalty term at all: the loss is the reconstruction error alone, and the sparsity is built into the encoder. In symbols,
$$f = \mathrm{TopK}_k\big(W_{\text{enc}}\,(x - b_{\text{dec}}) + b_{\text{enc}}\big).$$

How does this enforce sparsity, compared with L1?

- With L1, sparsity is a **soft preference**. Activity is taxed, and the network settles wherever the tax balances the reconstruction. How sparse the result is depends on $\lambda$, on the data and on the scale of the activations.
- With TopK, sparsity is a **hard rule**. The features compete, and for each input only the $k$ strongest are allowed to speak. We set L0 directly, by choosing $k$.

Since there is no penalty on the *size* of the activations, there is no shrinkage: a feature that wins is free to take whatever value reconstructs best.

One might worry that "keep the largest $k$" cannot be trained by gradient descent, since choosing the winners is a discrete decision. In practice it is no worse than a ReLU. For a given input, the winners are passed through unchanged, so gradient flows through them as normal, and the losers get no gradient, just like units on the flat side of a ReLU.

**Exercise 4.1.** Complete `topk_activation`. `torch.topk(pre, k, dim=1)` returns two tensors: the $k$ largest values in each row and their column positions. Start from a tensor of zeros and write the winning values into the right places with `scatter`.

In [ ]:
# Exercise 4.1
def topk_activation(pre, k):
    # pre: weighted sums, shape (N, n_features). Returns the same shape,
    # with the k largest entries of each row kept and all the others set to zero.
    values, positions = ...
    f = torch.zeros_like(pre).scatter(1, positions, values)     # put the winners back in place
    return F.relu(f)                                            # a winner should still not be negative


# a test: with k = 2 this should print [[0., 3., 0., 2.], [5., 0., 0., 1.]]
print(topk_activation(torch.tensor([[1., 3., -1., 2.], [5., 0., -2., 1.]]), 2))

The TopK autoencoder is the same as before except for the encoder, so we can reuse the class and replace one method.

In [ ]:
class TopKSparseAutoEncoder(SparseAutoEncoder):
    def __init__(self, k, d_in=2, n_features=10):
        super().__init__(d_in, n_features)
        self.k = k

    def encode(self, x):
        return topk_activation(self.pre_activations(x), self.k)

**Exercise 4.2.** Train TopK autoencoders with $k = 1$, $2$ and $3$, using `reconstruction_loss` (there is no penalty). Evaluate each, and collect the results in `topk_results`.

In [ ]:
# Exercise 4.2
topk_saes, topk_results = {}, {}
for k in [1, 2, 3]:
    torch.manual_seed(0)
    sae = ...
    topk_saes[k] = sae
    topk_results[k] = evaluate(sae)
    print(f"k = {k}", topk_results[k])

In [ ]:
plt.figure(figsize=(6, 4))
tradeoff_plot(l1_results, "L1 penalty (labels are $\\lambda$)")
tradeoff_plot(topk_results, "TopK (labels are $k$)", marker="s")
plt.legend();

**Question 4.1.** (a) Compare the TopK points with the L1 curve. At the same L0, which method reconstructs better, and why? (b) With $k = 2$ the reconstruction error is almost exactly zero. Why is that no surprise in our setting, and why would it not happen for a real layer with thousands of dimensions? (c) Give one advantage of each method over the other. (d) The $k = 2$ autoencoder has dead features. How can a feature die in a TopK autoencoder, and why can it not recover?

*Your answer:*

## 5. What did the features learn?

So far we have only asked whether the SAEs reconstruct well and how sparse they are. Now for the real question: **do the features mean anything?** In particular, since we gave the SAE ten features, has it found the ten digits?

We look at the TopK autoencoder with $k = 1$, the sparsest one: every input is described by a single feature.

### 5.1 Where is each feature active?

Each panel below shows the latent plane, with the test points coloured by how strongly one feature is active (grey means off). The cross is the SAE's origin $b_{\text{dec}}$, and the arrow is the feature's direction $d_j$.

In [ ]:
@torch.no_grad()
def feature_maps(sae, X=X_test, n_show=4000):
    f = sae.encode(X).numpy()
    pts = X.numpy()
    idx = np.random.default_rng(0).choice(len(pts), n_show, replace=False)
    origin = sae.b_dec.numpy()
    fig, axes = plt.subplots(2, 5, figsize=(17, 7))
    for j, ax in enumerate(axes.ravel()):
        on = f[idx, j] > 0
        ax.scatter(pts[idx][~on, 0], pts[idx][~on, 1], s=2, color="0.85", rasterized=True)
        ax.scatter(pts[idx][on, 0], pts[idx][on, 1], s=3, c=f[idx, j][on], cmap="viridis", rasterized=True)
        d = sae.W_dec[:, j].numpy()
        ax.annotate("", xy=origin + 1.5 * d, xytext=origin, arrowprops=dict(arrowstyle="->", color="red", lw=2))
        ax.plot(*origin, "r+", ms=10)
        ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"feature {j}   (active on {100 * (f[:, j] > 0).mean():.0f}% of inputs)", fontsize=10)
    plt.tight_layout()

sae1 = topk_saes[1]
feature_maps(sae1)

**Question 5.1.** (a) What shape is the region where a feature is active? (b) Explain that shape from the formulas for the encoder and the decoder, remembering that with $k = 1$ each point is reconstructed as $b_{\text{dec}} + f_j d_j$ for a single $j$. (c) Does the region depend on how far the point is from the origin, or only on its direction?

*Your answer:*

### 5.2 Which digits does each feature respond to?

Now we bring in the labels, purely to evaluate. The table below has a row for each feature and a column for each digit. Each entry is the percentage of that digit's test images on which the feature is active. A feature that had learned "this is a 3" would have a high number in column 3 and low numbers elsewhere.

In [ ]:
@torch.no_grad()
def feature_digit_table(sae, X=X_test, Y=Y_test):
    active = (sae.encode(X) > 0).numpy()
    return np.stack([active[Y == d].mean(axis=0) for d in range(10)], axis=1) * 100     # (features, digits)

def show_table(table, title):
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    im = ax.imshow(table, cmap="Blues", vmin=0, vmax=100)
    for j in range(table.shape[0]):
        for d in range(10):
            if table[j, d] >= 1:
                ax.text(d, j, f"{table[j, d]:.0f}", ha="center", va="center", fontsize=8,
                        color="white" if table[j, d] > 55 else "black")
    ax.set_xticks(range(10)); ax.set_yticks(range(table.shape[0]))
    ax.set_xlabel("digit"); ax.set_ylabel("feature"); ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, label="% of that digit's images on which the feature is active")

table1 = feature_digit_table(sae1)
show_table(table1, "TopK, k = 1")

A single number helps to summarise the table. Give each feature the label of the digit it is most often active for (using the training set). Then, for each test image, find its active feature and predict that feature's label. The fraction of correct predictions measures how much the features have to do with the digits. For comparison, the cell also reports:

- **chance**, the score for always guessing the commonest digit;
- **nearest neighbours in the latent plane**, a classifier that *is* given the labels, which shows how much information about the digit the two latent numbers contain at all.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

@torch.no_grad()
def feature_accuracy(sae):
    f_train, f_test = sae.encode(X_train).numpy(), sae.encode(X_test).numpy()
    top_train = np.where(f_train.max(axis=1) > 0, f_train.argmax(axis=1), -1)     # -1 means "no feature active"
    top_test = np.where(f_test.max(axis=1) > 0, f_test.argmax(axis=1), -1)
    label = {j: np.bincount(Y_train[top_train == j], minlength=10).argmax()
             for j in np.unique(top_train)}
    predicted = np.array([label.get(j, -1) for j in top_test])
    return (predicted == Y_test).mean(), label

acc1, labels1 = feature_accuracy(sae1)
knn = KNeighborsClassifier(15).fit(X_train.numpy(), Y_train).score(X_test.numpy(), Y_test)

print(f"chance:                                   {np.bincount(Y_test).max() / len(Y_test):.0%}")
print(f"most active SAE feature (TopK, k = 1):    {acc1:.0%}")
print(f"nearest neighbours in the latent plane:   {knn:.0%}")
print("label given to each feature:", {int(j): int(d) for j, d in labels1.items()})

**Question 5.2.** (a) Which digits have a feature (or several) that is mostly theirs? (b) Which digits have to share features with other digits? Compare with your answers to Question 1.1. (c) Several features have the same label. Why did the SAE spend more than one feature on the same digit? (d) The SAE scores well above chance and well below the nearest-neighbour classifier. What does each of those two gaps tell you?

*Your answer:*

### 5.3 What does a feature look like as an image?

Because the SAE's features live in the latent space of an autoencoder, we can do something that is not usually possible: turn a feature back into a picture. Take a feature's direction, walk outwards along it from the SAE's origin, and at each step pass the latent point through the *MNIST decoder* to get an image. Each row below is one feature; the activation increases from left to right.

In [ ]:
@torch.no_grad()
def decode_feature_directions(sae, strengths=np.linspace(0, 3, 9)):
    n = sae.W_dec.shape[1]
    fig, axes = plt.subplots(n, len(strengths), figsize=(0.9 * len(strengths), 0.9 * n))
    for j in range(n):
        for i, a in enumerate(strengths):
            x = sae.b_dec + float(a) * sae.W_dec[:, j]                  # a point in SAE units
            img = model.decode_images(to_latent(x).unsqueeze(0).to(device))[0].cpu()
            axes[j, i].imshow(img, cmap="gray_r", vmin=0, vmax=1)
            axes[j, i].set_xticks([]); axes[j, i].set_yticks([])
            if j == 0:
                axes[j, i].set_title(f"{a:.1f}", fontsize=7)
        axes[j, 0].set_ylabel(f"{j}", rotation=0, labelpad=10, va="center")
    fig.suptitle("Decoded images along each feature direction (activation increasing left to right)", fontsize=9)
    plt.tight_layout()

decode_feature_directions(sae1)

**Question 5.3.** (a) The first column is the same in every row. Why? (b) Find a row in which the image passes through one digit and then turns into another as the activation grows. Which digits, and how does this fit with Question 1.1(b)? (c) Putting Sections 5.1 to 5.3 together: what kind of structure in the latent space *can* this SAE pick out, and what kind can it not? Why did it not find ten features for ten digits?

*Your answer:*

### 5.4 The L1 autoencoder

Finally, compare with an L1 autoencoder of similar sparsity.

**Exercise 5.1.** Choose the value of $\lambda$ from Section 3 whose L0 is closest to 1.5. Draw its feature maps and its feature-by-digit table, and compute its accuracy.

In [ ]:
# Exercise 5.1
sae_l1 = l1_saes[...]              # choose a value of lambda
# then: feature_maps, show_table(feature_digit_table(...), ...), feature_accuracy

**Question 5.4.** How do the L1 features differ from the TopK features? Mention the shape of the active regions, what happens near the origin, and the dead features.

*Your answer:*

## 6. Why the real thing is different: high dimensions

Our SAE found directions, because directions are all it can represent, and in a plane there is not much room for directions. Ten of them, spread evenly round a circle, are 36° apart. Neighbouring features are almost parallel: the cosine of the angle between them is 0.81. Features that close together interfere. Moving along one moves you most of the way along its neighbour too, and the SAE cannot tell cleanly which of them an input belongs to.

In the layers where SAEs are really used, the situation is completely different, and the reason is a remarkable fact about high-dimensional space. The next cell picks random directions in spaces of different dimension and measures the cosine of the angle between every pair. A cosine of 0 means the two directions are at right angles (**orthogonal**); a cosine of 1 or $-1$ means they are parallel.

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, d in zip(axes, [2, 10, 100, 1000]):
    V = rng.normal(size=(500, d))
    V /= np.linalg.norm(V, axis=1, keepdims=True)          # 500 random directions of length 1
    cos = (V @ V.T)[np.triu_indices(500, k=1)]             # cosine for every pair
    ax.hist(cos, bins=60, range=(-1, 1))
    ax.set_title(f"{d} dimensions: typical |cosine| = {np.abs(cos).mean():.2f}", fontsize=10)
    ax.set_xlabel("cosine of angle between two random directions")
    ax.set_yticks([])
plt.tight_layout()

In two dimensions two random directions can be at any angle. In a thousand dimensions, *every* pair of random directions is very nearly at right angles. There are only $d$ directions that are exactly orthogonal to one another, but the number of directions that are *nearly* orthogonal grows enormously with $d$: far faster than $d$ itself.

This is what makes superposition, and therefore SAEs, work.

- A layer with a few thousand dimensions can hold many more than a few thousand feature directions, all nearly orthogonal to one another.
- Because they are nearly orthogonal, the features hardly interfere. Moving along one direction barely changes the position along any other, so an encoder can read off each feature cleanly.
- The small amount of interference that remains is harmless as long as only a few features are active at a time, which is exactly the sparsity that the SAE assumes and enforces.

So in a real application the SAE's dictionary is much *larger* than the layer, not five times but often hundreds of times, and the directions it finds are nearly orthogonal, each with room of its own. SAEs with an L1 penalty were used in this way to find readable features in language models (Cunningham et al., 2023; Bricken et al., 2023), and have since been trained with millions of features on the layers of large models (Templeton et al., 2024). The TopK version was introduced for the same purpose because it scales better and avoids shrinkage (Gao et al., 2024). Features found this way correspond to recognisable things, such as a particular city, a programming language, or text written in a certain tone.

Our two-dimensional layer has none of this room. That is why some digits could be given a direction of their own and others could not.

**Question 6.1.** (a) Read off from the histograms the typical size of the cosine between two random directions in 2 and in 1000 dimensions. (b) In our SAE, ten feature directions had to share a plane. Explain, using your answers from Section 5, how the lack of room showed up in the features. (c) Suppose the MNIST autoencoder had a latent layer with 1000 dimensions in place of 2. Give one reason to expect an SAE to find digit features more easily, and one reason why a wide latent layer might make the question itself less interesting.

*Your answer:*

## Summary

- A sparse autoencoder is trained on the **activations of a layer** of another network. It rewrites each activation vector as an origin plus a sum of **feature directions**, each multiplied by a non-negative activation.
- It is **wider** than the layer it reads. What makes it learn something is not a bottleneck but **sparsity**: only a few features may be active for any input.
- An **L1 penalty** taxes the total activation. It produces exact zeros because it pushes just as hard on small activations as on large ones (soft thresholding), at the cost of shrinking the activations it keeps. The feature directions must have fixed length, or the penalty can be dodged.
- **TopK** keeps the $k$ largest activations and zeroes the rest. Sparsity is a hard rule, set directly, with no shrinkage.
- SAEs are judged by **reconstruction error**, **L0** and **dead features**, which trade off against each other, and in the end by whether the features mean anything.
- An SAE can only find features that are **directions**. In our plane, digits with a direction of their own got features and digits that share a direction did not.
- Real SAEs work in spaces with thousands of dimensions, where enormous numbers of **nearly orthogonal** directions are available.

## If you have time

1. **More or fewer features.** Train TopK autoencoders with $k = 1$ and 4, 20 and 50 features. How do the wedges change? Does the accuracy keep improving?
2. **A different start.** Train the $k = 1$ autoencoder again with `seed=1` and `seed=2` in both `torch.manual_seed` and `train_sae`. Which features appear every time? What does this tell you about how far to trust any single feature?
3. **Reviving dead features.** The $k = 2$ TopK autoencoder has dead features. Try to bring them back: for example, every few epochs, reset the encoder weights of any dead feature to point at an input that is currently reconstructed badly.
4. **A wider layer.** The first layer of the MNIST decoder has 256 units and is computed from the latent layer. Use a forward hook to collect its activations, and train an SAE on them with `d_in=256`. Do the features look more like digits?